# UA1 – Bloc 2 (RAG) – Notebook (Démo du cours) 

Objectif (UA1) : **Assistant RAG** qui répond **factuellement** à partir des PDFs de l’hôtel, avec **sources**, et tests de **prompting** (zero-shot / few-shot / CoT). fileciteturn28file0

---

## Structure du projet attendue

- `src/data/rag_docs/`  → PDFs (base de connaissances)
- `src/data/chroma_db/` → base vectorielle Chroma (persistée)
- `outputs/rag_logs/`   → logs de conversations (jsonl)


In [ ]:
# (2) Imports + chemins (adaptés à ton projet)

from pathlib import Path
import json, re
import requests
from datetime import datetime, timezone

# Détection du "project root" 
def find_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(8):
        if (p / "src").exists():
            return p
        p = p.parent
    return start.resolve()

ROOT = find_root(Path.cwd())

DOCS_DIR = ROOT / "src" / "data" / "rag_docs"
PERSIST_DIR = ROOT / "src" / "data" / "chroma_db"
LOG_DIR = ROOT / "outputs" / "rag_logs"
LOG_PATH = LOG_DIR / "rag_logs.jsonl"

# Modèles Ollama 
GEN_MODEL = "gemma3:1b"
EMBED_MODEL = "nomic-embed-text:latest"

OLLAMA_BASE = "http://localhost:11434"

print("ROOT       =", ROOT)
print("DOCS_DIR   =", DOCS_DIR)
print("PERSISTDIR =", PERSIST_DIR)
print("LOG_PATH   =", LOG_PATH)

ROOT       = C:\Users\Administrator\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Intelligence Artificielle Générative\projet_UA1
DOCS_DIR   = C:\Users\Administrator\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Intelligence Artificielle Générative\projet_UA1\src\data\rag_docs
PERSISTDIR = C:\Users\Administrator\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Intelligence Artificielle Générative\projet_UA1\src\data\chroma_db
LOG_PATH   = C:\Users\Administrator\OneDrive\Bureau\@ProgrammeAI_cours_session_04\Intelligence Artificielle Générative\projet_UA1\outputs\rag_logs\rag_logs.jsonl


In [ ]:
# (3) Vérifier que Ollama répond + modèles dispo

r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=10)
r.raise_for_status()
models = [m["name"] for m in r.json().get("models", [])]
print("Ollama OK")
print("Modèles:", models)

assert any(GEN_MODEL in m for m in models), f"⚠️ Modèle gen absent: {GEN_MODEL}"
assert any(EMBED_MODEL in m for m in models), f"⚠️ Modèle embed absent: {EMBED_MODEL}"


✅ Ollama OK
Modèles: ['gemma3:4b', 'nomic-embed-text:latest', 'gemma3:1b']


## Étape A — Charger les PDFs (PDF Loader)

On charge chaque page comme un `Document` et on conserve `source` + `page` dans les métadonnées.


In [ ]:
# Loader PDF
from langchain_community.document_loaders import PyPDFLoader

pdfs = sorted(DOCS_DIR.glob("*.pdf"))
assert pdfs, f"Aucun PDF trouvé dans: {DOCS_DIR}"

docs = []
for pdf in pdfs:
    loader = PyPDFLoader(str(pdf))
    loaded = loader.load()  # une page = un Document
    for d in loaded:
        d.metadata["source"] = pdf.name
    docs.extend(loaded)

print(f"PDFs: {len(pdfs)}")
print(f"Pages (docs): {len(docs)}")
print("Exemple metadata:", docs[0].metadata)

✅ PDFs: 7
✅ Pages (docs): 51
Exemple metadata: {'producer': 'Microsoft® Word pour Microsoft\xa0365', 'creator': 'Microsoft® Word pour Microsoft\xa0365', 'creationdate': '2024-06-08T11:19:29-04:00', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_enabled': 'true', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_setdate': '2024-05-07T15:05:21Z', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_method': 'Standard', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_name': 'defa4170-0d19-0005-0004-bc88714345d2', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_siteid': 'ad8a84ef-f1f3-4b14-ad08-b99ca66f7e30', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_actionid': '71bec76f-3028-4248-b7fc-051c767273fa', 'msip_label_3513b79f-ce8d-43d6-b7e5-c12d3e236916_contentbits': '0', 'author': 'Julie Sévigny', 'moddate': '2024-06-08T11:19:29-04:00', 'source': 'Convention collective - services alimentaires.pdf', 'total_pages': 27, 'page': 0, 'page_label': '1'}


## Étape B — Chunking (RecursiveCharacterTextSplitter)

Paramètres de démo du cours (simple & efficace) :
- `chunk_size = 500`
- `chunk_overlap = 50`


In [ ]:
# Splitter (compat)
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except Exception:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

print("Chunks (splits):", len(splits))
print("Exemple chunk:", len(splits[0].page_content), "chars")
print("Exemple source/page:", splits[0].metadata.get("source"), splits[0].metadata.get("page"))

✅ Chunks (splits): 218
Exemple chunk: 262 chars
Exemple source/page: Convention collective - services alimentaires.pdf 0


## Étape C — Embeddings (Ollama)

On utilise un modèle **d’embeddings** dédié : `nomic-embed-text`.


In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

# Test rapide
v = embeddings.embed_query("test embedding")
print("Embedding OK | dim =", len(v))

✅ Embedding OK | dim = 768


## Étape D — Vector DB (Chroma persisté)

La DB est persistée dans : `src/data/chroma_db/`.

### Si tu veux reconstruire (ex: nouveau PDF ajouté)
Décommente la partie `shutil.rmtree(PERSIST_DIR)` puis relance la cellule.


In [ ]:
# Vectorstore Chroma (compat)
try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

# (Option) Repartir à zéro 
# import shutil
# if PERSIST_DIR.exists():
#     shutil.rmtree(PERSIST_DIR)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory=str(PERSIST_DIR),
)

# Retriever (MMR = diversité, moins de doublons)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 25, "lambda_mult": 0.5}
)

In [7]:
def dedup_docs(docs):
    seen = set()
    out = []
    for d in docs:
        key = (d.metadata.get("source"), d.metadata.get("page"), d.page_content[:120])
        if key in seen:
            continue
        seen.add(key)
        out.append(d)
    return out

## Étape E — RAG Chain + Prompts (zero-shot / few-shot / CoT)

UA1 demande de tester des stratégies de prompting. fileciteturn28file0

On garde un prompt **strict** (anti-hallucination) :
- Répond uniquement avec le contexte
- Sinon : *Je ne sais pas (info absente des documents).*

Puis on teste 3 variantes :
1) **zero_shot** (simple)
2) **few_shot** (2 mini exemples)
3) **cot** (raisonnement demandé, réponse finale courte)


In [ ]:
# Étape E (code) — RAG: retrieve + prompts + ask() + logs 

# ChatOllama (compat)
try:
    from langchain_community.chat_models import ChatOllama
except Exception:
    from langchain_ollama import ChatOllama

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Si gemma3:4b donne "not enough memory", on reste sur gemma3:1b
llm = ChatOllama(model=GEN_MODEL, temperature=0.0)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

# Routing simple (évite les PDFs hors-sujet) 
SOURCE_RULES = {
    "faq": {
        "Partie 3 - FAQ - Hôtel De la Promenade.pdf",
        "politiques_hotel_de_la_promenade_document.pdf",
    },
    "sst": {"Membres du comité SST.pdf"},
    "reservation_process": {"Processus - Gestion des réservations.pdf"},
}

def detect_intent(q: str) -> str:
    ql = q.lower()
    if any(k in ql for k in ["sst", "comité", "comite"]):
        return "sst"
    if any(k in ql for k in ["processus", "réservation", "reservation", "réservations", "reservations"]):
        return "reservation_process"
    if any(k in ql for k in [
        "check-in", "check out", "check-out", "enregistrement", "départ", "checkout",
        "annulation", "tarif flexible", "no-show",
        "animaux", "animal", "petit-déjeuner", "dejeuner", "déjeuner", "parking", "wifi"
    ]):
        return "faq"
    return "general"

def retrieve(question: str, k_final: int = 4, k_candidates: int = 12):
    # 1) MMR via Chroma (plus de diversité)
    # 2) dédup
    # 3) filtre par intent
    docs = vectorstore.max_marginal_relevance_search(
        question, k=k_candidates, fetch_k=25, lambda_mult=0.5
    )
    docs = dedup_docs(docs)

    intent = detect_intent(question)
    allowed = SOURCE_RULES.get(intent)
    if allowed:
        docs = [d for d in docs if d.metadata.get("source") in allowed]

    return docs[:k_final]

def format_docs(docs):
    out = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "?")
        out.append(f"[{src} | page {page}]\n{d.page_content}")
    return "\n\n".join(out)

# Prompts UA1 (3 variantes)
BASE_RULES = '''Tu es un assistant RAG STRICT.

Règles:
1) Réponds uniquement avec les informations présentes dans le CONTEXTE.
2) Pas de style marketing (pas de bienvenue, pas de questions).
3) Réponse courte (max 2 phrases).
4) Si une info manque: écris exactement "Je ne sais pas (info absente des documents)."
5) Si la question demande check-in ET check-out:
   réponds exactement: "Check-in: <heure>. Check-out: <heure>."
'''

ZERO_SHOT = BASE_RULES + '''
CONTEXTE:
{context}

QUESTION: {question}
'''

FEW_SHOT = BASE_RULES + '''
Exemples (format attendu):
Q: À quelle heure est le check-in et le check-out ?
A: Check-in: 15h00. Check-out: 11h00.

Q: Quelle est la politique d’annulation pour un tarif flexible ?
A: Annulation gratuite jusqu’à 24 h avant 15h00 le jour d’arrivée.

CONTEXTE:
{context}

QUESTION: {question}
'''

COT = BASE_RULES + '''
Important: réfléchis en interne si nécessaire, mais n’affiche PAS ton raisonnement.
Donne seulement la réponse finale.

CONTEXTE:
{context}

QUESTION: {question}
'''

parser = StrOutputParser()
chains = {
    "zero_shot": ChatPromptTemplate.from_template(ZERO_SHOT) | llm | parser,
    "few_shot": ChatPromptTemplate.from_template(FEW_SHOT) | llm | parser,
    "cot": ChatPromptTemplate.from_template(COT) | llm | parser,
}

def _postcheck(question: str, answer: str) -> str:
    ql = question.lower()
    # si la question exige check-in + check-out, on refuse une réponse incomplète
    if ("check-in" in ql or "enregistrement" in ql) and ("check-out" in ql or "départ" in ql or "check out" in ql):
        if ("check-in" not in answer.lower()) or ("check-out" not in answer.lower()):
            return "Je ne sais pas (info absente des documents)."
    return answer

def ask(question: str, prompt_name: str = "zero_shot", show_sources: bool = True) -> str:
    docs = retrieve(question, k_final=4, k_candidates=12)

    if show_sources:
        print("Sources (retrieval):")
        for d in docs:
            print(f"* {d.metadata.get('source')} | page {d.metadata.get('page')}")
        print()

    context = format_docs(docs)
    answer = chains[prompt_name].invoke({"context": context, "question": question}).strip()
    answer = _postcheck(question, answer)

    # log jsonl (UA1) 
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    entry = {
        "timestamp": utc_now(),
        "gen_model": GEN_MODEL,
        "embed_model": EMBED_MODEL,
        "prompt": prompt_name,
        "question": question,
        "answer": answer,
        "sources": [
            {"source": d.metadata.get("source"), "page": d.metadata.get("page")}
            for d in docs
        ],
    }
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    return answer


C:\Users\Administrator\AppData\Local\Temp\ipykernel_22012\2872098018.py:13: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=GEN_MODEL, temperature=0.0)


## Étape G — Tests rapides

In [10]:
print(ask("À quelle heure est le check-in et le check-out ?"))
print(ask("Quelle est la politique d’annulation pour un tarif flexible ?"))

Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 3
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 3
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 2
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0

Check-in: 15h00. Check-out: <heure>
Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 5
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 2

Nous comprenons que les plans peuvent évoluer comme les saisons. Pour nos tarifs flexibles, vous pouvez annuler sans pénalité jusqu'à 24 heures avant 15h00 le jour de votre arrivée prévue — une liberté


## Étape H — Comparaison prompts (UA1)

In [11]:
question = "Quelle est la politique d’annulation pour un tarif flexible ?"

for p in ["zero_shot", "few_shot", "cot"]:
    print("="*80)
    print("PROMPT:", p)
    print("R:", ask(question, prompt_name=p, show_sources=True))

PROMPT: zero_shot
Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 5
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 2

R: Nous comprenons que les plans peuvent évoluer comme les saisons. Pour nos tarifs flexibles, vous pouvez annuler sans pénalité jusqu'à 24 heures avant 15h00 le jour de votre arrivée prévue — une liberté
PROMPT: few_shot
Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 5
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 2

R: R : Nous comprenons que les plans peuvent évoluer comme les saisons. Pour nos tarifs flexibles, vous
pouvez annuler sans pénalité jusqu’à 24 heures avant 15h00 le jour de votre arrivée prévue — une liberté
PROMPT: cot
Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 5
* Partie 3 - FAQ - Hôtel De la Promenade.p

## Étape I — Pack de questions (10–15 conversations)

In [ ]:
QUESTIONS = [
    "Que se passe-t-il en cas de No-Show ?",
    "Les animaux de compagnie sont-ils acceptés ? Quels frais ?",
    "Qui sont les membres du comité SST et comment les contacter ?",
    "C’est quoi le problème du processus de réservation actuel ?",
]

for q in QUESTIONS:
    print("="*80)
    print("Q:", q)
    print("A:", ask(q, prompt_name="zero_shot"))

print("\nLogs écrits dans:", LOG_PATH)

Q: Que se passe-t-il en cas de No-Show ?
Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 2
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 5
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 3

A: Je ne sais pas (No-Show).
Q: Les animaux de compagnie sont-ils acceptés ? Quels frais ?
Sources (retrieval):
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 1
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 0
* politiques_hotel_de_la_promenade_document.pdf | page 6
* Partie 3 - FAQ - Hôtel De la Promenade.pdf | page 3

A: Oui, les animaux de compagnie sont admis sans frais. Les frais de nettoyage sont de 50$ par séjour (non remboursable) ou frais de 35$/nuit selon le forfait. Le dépôt remboursable de 150$ est pour possibles dommages.
Q: Qui sont les membres du comité SST et comment les contacter ?
Sources (retrieval):

A: Je ne sais pas (informations sur les membres du comité SST).
Q: C’est quoi le pr